# RG3 gas derived-feature combinations
Fourteen input configurations on all shots and a common complete-history cohort. Fixed Random Forest, block holdout evaluation.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, f1_score
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
tables=root/'reports/tables'
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
prep=''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source'])
exec(prep.split('for feature in features:')[0])
sensors=['Average_Back_Pressure','Max_Injection_Pressure','Cushion_Position','Barrel_Temperature_5']
for f in sensors:
    prev=observed.groupby('history_segment')[f].shift()
    grouped=prev.groupby(observed.history_segment)
    observed[f+'__mean10']=grouped.transform(lambda s:s.rolling(10,min_periods=10).mean())
    for w in [100,300]:
        mean=grouped.transform(lambda s:s.rolling(w,min_periods=w).mean())
        observed[f+'__dev'+str(w)]=observed[f]-mean
        observed[f+'__trend'+str(w)]=observed[f+'__mean10']-mean
shots=label_shots.merge(observed,on=keys,how='left',validate='one_to_one')
data=shots.loc[shots['product'].eq('RG3') & (shots.PassOrFail.eq(0)|shots.Reason.str.contains('가스',regex=False))].reset_index(drop=True)
y=data.Reason.str.contains('가스',regex=False).astype(int)
assert len(data)==583 and y.sum()==17 and data.comparison_block.nunique()==5
p1,p2,c,t=sensors
configs={}
for w in [100,300]:
    a=p1+'__dev'+str(w);b=p2+'__dev'+str(w);cc=c+'__dev'+str(w);d=p1+'__trend'+str(w);e=t+'__trend'+str(w)
    configs.update({f'A_{w}':[a,b],f'B_{w}':[a,b,cc],f'C_{w}':[a,b,e],f'D_{w}':[a,b,d],f'E_{w}':[a,b,cc,e]})
configs.update({'raw_A':[p1,p2],'raw_B':[p1,p2,c],'raw_C':[p1,p2,t],'raw_E':[p1,p2,c,t]})
coverage=[]
common=data[[f+'__dev300' for f in sensors]].notna().all(axis=1)
print('common cohort',int(common.sum()),'gas',int(y[common].sum()),flush=True)
for name,cols in configs.items():
    good=data[cols].notna().all(axis=1)
    coverage.append(dict(setting=name,features=len(cols),complete_shots=int(good.sum()),complete_gas=int(y[good].sum()),missing_shots=int((~good).sum()),missing_gas=int(y[~good].sum())))
# Use exactly the previously audited evaluation function (no previous experiments executed).
nb13=json.loads((root/'notebooks/13_rg3_history_window_comparison.ipynb').read_text(encoding='utf-8'))
src=''.join(next(c for c in nb13['cells'] if c['cell_type']=='code')['source'])
results=[];predictions=[];byblock=[]
exec('def evaluate('+src.split('def evaluate(',1)[1].split('# Exact former baseline',1)[0])
for cohort,mask in [('all_with_train_imputation',pd.Series(True,index=data.index)),('common_complete300',common)]:
    frame=data.loc[mask].reset_index(drop=True); target=y.loc[mask].reset_index(drop=True)
    for name,cols in configs.items():
        nblock=len(byblock)
        evaluate(frame,target,cols,int(name.split('_')[-1]) if not name.startswith('raw') else 0,cohort)
        results[-1]['setting']=name;results[-1]['features']=len(cols)
        predictions[-1]['setting']=name
        for row in byblock[nblock:]:row['setting']=name
        # Checkpoint results to make progress auditable while running.
        pd.DataFrame(results).to_csv(tables/'rg3_derived_combo_metrics.csv',index=False,encoding='utf-8-sig')
metrics=pd.DataFrame(results)
pd.DataFrame(coverage).to_csv(tables/'rg3_derived_combo_coverage.csv',index=False,encoding='utf-8-sig')
pred=pd.concat(predictions,ignore_index=True)
assert not pred.duplicated(['cohort','setting']+keys).any()
assert pred.probability.notna().all()
assert (metrics.TP+metrics.FN==metrics.gas).all()
assert (metrics[['TP','FP','FN','TN']].sum(axis=1)==metrics.shots).all()
pred.to_csv(tables/'rg3_derived_combo_oof_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(byblock).to_csv(tables/'rg3_derived_combo_by_block.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([dict(setting=k,features=len(v),columns=';'.join(v)) for k,v in configs.items()]).to_csv(tables/'rg3_derived_combo_inputs.csv',index=False,encoding='utf-8-sig')
data[keys+['comparison_block','Reason']+list(dict.fromkeys(c for cols in configs.values() for c in cols))].assign(gas=y).to_csv(tables/'rg3_derived_combo_dataset.csv',index=False,encoding='utf-8-sig')
# Paired comparisons to distinguish adding sensors from deriving features.
pairs=[]
for cohort in metrics.cohort.unique():
    sub=metrics.loc[metrics.cohort.eq(cohort)].set_index('setting')
    for w in [100,300]:
        for letter,baseline in [('A','raw_A'),('B','raw_B'),('C','raw_C'),('D','raw_A'),('E','raw_E')]:
            derived=sub.loc[f'{letter}_{w}']; raw=sub.loc[baseline]
            pairs.append(dict(cohort=cohort,setting=f'{letter}_{w}',raw_baseline=baseline,delta_AP=derived.AP-raw.AP,delta_caught_top10=int(derived.caught_top10-raw.caught_top10),delta_FP=int(derived.FP-raw.FP)))
pd.DataFrame(pairs).to_csv(tables/'rg3_derived_combo_vs_raw.csv',index=False,encoding='utf-8-sig')
lines=['# RG3 가스: 소수 파생변수 조합 비교','',
'정상566 Shot과 가스17 Shot. RH/LH 합친 고유Shot 기준. 순수 미성형8개 제외. 기존 실험과 같은 연속 관측 이력과 관측 구간 평가를 사용했다.',
'','## 입력 조합','',
'기준평균은 현재를 제외한 직전100 또는300개 관측Shot 평균이다. 최근10개 평균도 현재를 제외한다. 창 길이를 모두 채우지 못하면 결측이다. 편차의 부호를 유지했다.',
'','| 조합 | 입력 | 수 |','|---|---|---:|',
'| A | 현재 평균배압−장기평균, 현재 최대사출압력−장기평균 | 2 |',
'| B | A + 현재 쿠션위치−장기평균 | 3 |',
'| C | A + 최근10개 배럴5온도평균−장기평균 | 3 |',
'| D | A + 최근10개 평균배압평균−장기평균 | 3 |',
'| E | A + 쿠션편차 + 배럴5온도추세 | 4 |',
'| raw_A/B/C/E | 해당 조합의 원래 센서값만 사용. D는 raw_A와 비교 | 2/3/3/4 |',
'','## 평가 조건','',
'Random Forest: 200트리, 깊이8, 말단최소3, balanced_subsample, 시드42. 구간하나를 통째로 제외하는5회 평가. 구간은 실제Run ID가 아니다. Timestamp/구간ID/불량이유는 입력에서 제외했다.',
'',
'전체583 Shot에서는 결측을 각 학습구간 중앙값으로 대체했다. 동일완전이력 비교에서는 네센서 모두 이전300개 이력이 있는 동일Shot만 학습·평가했다. 비라벨은 센서이력에만 사용하고 정상으로 간주하지 않았다. 생산/정지 상태를 별도 필터링하지 않아 긴기준선에 비생산기록이 섞일 가능성은 추가 점검 과제다.',
'',
'0.5 임계값은 고정했다. 상위약10%는 각 평가구간별 위험순위로 선택하고 동률은 고정시드로 처리했다. 미래구간이 학습에 포함될 수 있으며 실제미래평가와 다르다. 조합선택에 같은평가결과를 사용하므로 독립최종성능으로 해석하지 않는다.']
for cohort in metrics.cohort.unique():
    sub=metrics.loc[metrics.cohort.eq(cohort)]
    lines+=['','## '+('전체 Shot, 이력 부족은 대체' if cohort.startswith('all') else '동일한 완전이력 Shot 비교'),'',
    f'전체 {int(sub.iloc[0].shots)} Shot, 가스 {int(sub.iloc[0].gas)} Shot.',
    '','| 조합 | 변수수 | TP | FP | FN | TN | AP | F1 | 상위10% 발견 | 검사수 |','|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
    for r in sub.itertuples():lines.append(f'| {r.setting} | {r.features} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.AP:.4f} | {r.F1:.4f} | {r.caught_top10}/{r.gas} | {r.selected_top10} |')
lines+=['','## 원값 대비 파생변수 효과','',
'동일센서 원값 모델과의 차이. 양수AP/발견수 차이는 개선 방향, 양수오탐 차이는 악화 방향. 단일실험의 소표본 탐색결과다.',
'','| 평가대상 | 조합 | 원값기준 | AP 차이 | 상위10% 발견수 차이 | FP 차이 |','|---|---|---|---:|---:|---:|']
for r in pd.DataFrame(pairs).itertuples():lines.append(f'| {r.cohort} | {r.setting} | {r.raw_baseline} | {r.delta_AP:+.4f} | {r.delta_caught_top10:+d} | {r.delta_FP:+d} |')
(root/'reports/rg3_derived_combo_comparison.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('VERIFIED 28 experiments; files saved.',flush=True)



common cohort 336 gas 10


C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-pa

C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-pa

C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-pa

C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\hayeon\workspace\KAMP-IMM\.venv\Lib\site-pa

...

{'window': 0, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 2, 'FP': 35, 'FN': 8, 'TN': 291, 'precision': 0.05405405405405406, 'recall': 0.2, 'F1': 0.0851063829787234, 'AP': 0.04824270598245254, 'selected_top10': 36, 'caught_top10': 2}


{'window': 0, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 2, 'FP': 15, 'FN': 8, 'TN': 311, 'precision': 0.11764705882352941, 'recall': 0.2, 'F1': 0.14814814814814814, 'AP': 0.08990407530924116, 'selected_top10': 36, 'caught_top10': 3}


{'window': 0, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 2, 'FP': 10, 'FN': 8, 'TN': 316, 'precision': 0.16666666666666666, 'recall': 0.2, 'F1': 0.18181818181818182, 'AP': 0.09549396932431109, 'selected_top10': 36, 'caught_top10': 3}


{'window': 0, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 2, 'FP': 6, 'FN': 8, 'TN': 320, 'precision': 0.25, 'recall': 0.2, 'F1': 0.2222222222222222, 'AP': 0.1544947909969831, 'selected_top10': 36, 'caught_top10': 3}


VERIFIED 28 experiments; files saved.
